In [ ]:
import phoebe
import sys
sys.path.insert(1, '../scripts')
import phoebe_to_eBEER
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from scipy.optimize import curve_fit

import itertools
import time
import pickle

In [ ]:
eccs = (0, 0.3, 0.8)
incls = (0, 30, 60, 90, 120, 150, 180)
per0s = (0, 30, 90, 150, 180, 240)
qs = (1, 0.5, 2,)
ps = (1, 5, 10, 15, 20,)

fn = ""
Refl_ON = False

In [ ]:
with open('{fn}.pkl', 'rb') as f:
    loaded_dict = pickle.load(f)
print(len(loaded_dict))

comb = itertools.product(eccs, incls, per0s, ps, qs)
comb_list = list(comb)
print(len(comb_list))

In [ ]:
periods = np.zeros(len(comb_list))
PHOEBE_A = np.zeros(len(comb_list))
eBEER_A = np.zeros(len(comb_list))
RMSDs = np.zeros(len(comb_list))
for i in range(len(comb_list)):
    comb = comb_list[i]
    ecc, incl, per0, p, q = comb
    comb_str = f'lc_{p}_{int(ecc*10)}_{incl}_{per0}_{int(q*10)}'

    try:
        if not Refl_ON:
            times, t0, relative_flux, eBEER_flux = loaded_dict[comb_str]
        else:
            times, t0, relative_flux, fit_flux, rescaled = loaded_dict[comb_str]
    except:
        print(f'Failed: {i} -', comb_str)
        continue

    if not Refl_ON: residuals = (relative_flux - eBEER_flux)
    else: residuals = (relative_flux - fit_flux)
    
    MSD = np.mean(residuals**2)
    RMSD = np.sqrt(MSD)
    RMSDs[i] = RMSD
    periods[i] = p

    if not Refl_ON:
        eBEER_A[i] = (np.max(eBEER_flux) - np.min(eBEER_flux)) / 2
    else:
        eBEER_A[i] = (np.max(fit_flux) - np.min(fit_flux)) / 2
    PHOEBE_A[i] = (np.max(relative_flux) - np.min(relative_flux)) / 2

In [ ]:
plt.scatter(periods, RMSDs)
plt.xscale('log')
plt.xlabel('Period (days)')
plt.ylabel('RMSD (ppm)')
plt.title('sys_config')
plt.legend()

In [ ]:
plt.plot(periods, eBEER_A, 'b.', label= 'eBEER')
plt.plot(periods, PHOEBE_A, 'r.', label= 'PHOEBE')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Period (days)')
plt.ylabel('Amplitude (ppm)')
plt.title('LC Amplitude vs Orbital Period')
plt.legend()